## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-07-04-41-28 +0000,nyt,2 Skydivers Die After Midair Collision in Sout...,https://www.nytimes.com/2026/11/06/us/californ...
1,2026-11-07-04-31-20 +0000,nypost,Runaway piglets lead South Carolina cops on hi...,https://nypost.com/2026/11/06/us-news/runaway-...
2,2026-11-07-04-31-00 +0000,wsj,China Forex Reserves Edged Up in October,https://www.wsj.com/economy/central-banking/ch...
3,2026-11-07-04-27-35 +0000,startribune,Gophers’ depth tested as St. Thomas makes hist...,https://www.startribune.com/minnesota-gophers-...
4,2026-11-07-04-16-39 +0000,startribune,Wolves outlast Pelicans for third straight vic...,https://www.startribune.com/minnesota-timberwo...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2463/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
155,trump,55
173,new,28
199,miller,18
198,stephen,17
200,chief,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
106,2026-11-06-22-36-00 +0000,wsj,President Trump named Stephen Miller his new c...,https://www.wsj.com/politics/stephen-miller-is...,222
32,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...,221
113,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,189
207,2026-11-06-19-07-29 +0000,cbc,Trump names Stephen Miller as new chief of sta...,https://www.cbc.ca/news/world/trump-susie-wile...,187
195,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...,187


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
106,222,2026-11-06-22-36-00 +0000,wsj,President Trump named Stephen Miller his new c...,https://www.wsj.com/politics/stephen-miller-is...
20,81,2026-11-07-03-00-00 +0000,wsj,Elon Musk-run companies like SpaceX and Tesla ...,https://www.wsj.com/politics/policy/elon-musk-...
98,80,2026-11-06-23-00-02 +0000,chitri,La Shawn K. Ford vs Chad Koppie: How Illinois’...,https://www.chicagotribune.com/2026/11/06/la-s...
291,76,2026-11-06-15-32-25 +0000,chitri,Father of Meghan Markle arrested on charges re...,https://www.chicagotribune.com/2026/11/06/megh...
298,69,2026-11-06-15-22-00 +0000,wsj,The solid October jobs report leaves inflation...,https://www.wsj.com/economy/central-banking/so...
113,69,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...
153,51,2026-11-06-21-13-25 +0000,nyt,Kenya Cult Leader Handed 200 Year Prison Sente...,https://www.nytimes.com/2026/11/06/world/afric...
101,47,2026-11-06-22-49-25 +0000,latimes,Jane Kim wins race to be California's next ins...,https://www.latimes.com/business/story/2026-11...
31,45,2026-11-07-02-00-00 +0000,wsj,The wave election that gave Democrats Congress...,https://www.wsj.com/politics/elections/2028-el...
195,44,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
